## Question 5 — Encoder–decoder cross-attention

Self-attention draws $Q,K,V$ from one sequence. In cross-attention, $Q=H_{decoder}W_Q\in\mathbb R^{n\times d_k}$ and $K=H_{encoder}W_K\in\mathbb R^{m\times d_k}$, $V=H_{encoder}W_V\in\mathbb R^{m\times d_v}$. Scores and weights are $n\times m$; output is $n\times d_v$.

## Code Execution

In [6]:
import numpy as np

**Attention from Question 2.** Cross-attention calls this scaled dot-product function.

In [7]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """Q:(nq, dk), K:(nk, dk), V:(nk, dv) -> output (nq, dv), weights (nq, nk)."""
    Q, K, V = (np.asarray(z, dtype=float) for z in (Q, K, V))
    if Q.ndim != 2 or K.ndim != 2 or V.ndim != 2 or Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError("Incompatible Q, K, V dimensions")
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights

**Cross-attention.** Queries come from the decoder. Keys and values come from the encoder.

In [8]:
def cross_attention(decoder_states, encoder_states, W_Q, W_K, W_V):
    """decoder:(n, d_model), encoder:(m, d_model) -> output (n, d_v), weights (n, m)."""
    Q = decoder_states @ W_Q
    K = encoder_states @ W_K
    V = encoder_states @ W_V
    return scaled_dot_product_attention(Q, K, V)

**Check.** One-hot encoder states and the given matrices reproduce $K$ and $V$. Weights are $1\times 3$, and the output is about $[[1.203, 0.797]]$.

In [10]:
Q5 = np.array([[1.0, 0.0]])
K5 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V5 = np.array([[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
out5_direct, weights5_direct = scaled_dot_product_attention(Q5, K5, V5)

# One-hot encoder states, so W_K = K5 and W_V = V5 reproduce those matrices.
encoder_states5 = np.eye(3)
decoder_states5 = np.array([[1.0, 0.0, 0.0]])
WQ5 = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])
out5, weights5 = cross_attention(decoder_states5, encoder_states5, WQ5, K5, V5)

print("weights:\n", np.round(weights5, 6))
print("output:\n", np.round(out5, 6))
assert weights5.shape == (1, 3) and out5.shape == (1, 2)
np.testing.assert_allclose(weights5, weights5_direct)
np.testing.assert_allclose(out5, out5_direct)
np.testing.assert_allclose(out5_direct, [[1.203336, 0.796664]], atol=1e-6)
print("Cross-attention check passed.")

weights:
 [[0.401112 0.197776 0.401112]]
output:
 [[1.203336 0.796664]]
Cross-attention check passed.


(d) Each decoder token gathers a weighted mixture of source values that match its query. The matrix is rectangular when source length $m$ and target length $n$ differ.